In [0]:
%sql
USE CATALOG ecommerce_db

In [0]:
#Explain query
spark.sql("SELECT * FROM silver.events WHERE event_type='purchase'").explain(True)

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('event_type = purchase)
   +- 'UnresolvedRelation [silver, events], [], false

== Analyzed Logical Plan ==
event_time: timestamp, event_type: string, product_id: int, category_id: bigint, category_code: string, brand: string, price: double, user_id: int, user_session: string, ingestion_ts: timestamp, event_date: date, price_tier: string
Project [event_time#13208, event_type#13209, product_id#13210, category_id#13211L, category_code#13212, brand#13213, price#13214, user_id#13215, user_session#13216, ingestion_ts#13217, event_date#13218, price_tier#13219]
+- Filter (event_type#13209 = purchase)
   +- SubqueryAlias ecommerce_db.silver.events
      +- Relation ecommerce_db.silver.events[event_time#13208,event_type#13209,product_id#13210,category_id#13211L,category_code#13212,brand#13213,price#13214,user_id#13215,user_session#13216,ingestion_ts#13217,event_date#13218,price_tier#13219] parquet

== Optimized Logical Plan ==
Filter (isnotnull(

In [0]:
# Benchmark
import time
start = time.time()
spark.sql("SELECT * FROM silver.events WHERE DATE(event_time)='2019-11-01'").count()
print(f"Time: {time.time()-start:.2f}s")

Time: 4.94s


In [0]:
#Partitioned table
spark.sql("""
  CREATE TABLE silver.events_part
  USING DELTA
  PARTITIONED BY (event_date, event_type)
  AS SELECT * FROM silver.events
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Benchmark
import time
start = time.time()
spark.sql("SELECT * FROM silver.events_part WHERE DATE(event_time)='2019-11-01' and user_id = 546144740").count()
print(f"Time: {time.time()-start:.2f}s")

Time: 0.97s


In [0]:
# Optimize
spark.sql("OPTIMIZE silver.events_part ZORDER BY (user_id, product_id)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# Benchmark
import time
start = time.time()
spark.sql("SELECT * FROM silver.events_part WHERE DATE(event_time)='2019-11-01' and user_id = 546144740").count()
print(f"Time: {time.time()-start:.2f}s")

Time: 0.58s


In [0]:
# Cache for iterative queries
cached = spark.table("silver.events").cache()
cached.count()  # Materialize

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6891381662638264>, line 2
      1 # Cache for iterative queries
----> 2 cached = spark.table("silver.events").cache()
      3 cached.count()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2093, in DataFrame.cache(self)
   2092 def cache(self) -> ParentDataFrame:
-> 2093     return self.persist()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2100, in DataFrame.persist(self, storageLevel)
   2095 def persist(
   2096     self,
   2097     storageLevel: StorageLevel = (StorageLevel.MEMORY_AND_DISK_DESER),
   2098 ) -> ParentDataFrame:
   2099     relation = self._plan.plan(self._session.client)
-> 2100     self._session.client._analyze(
   2101         method="persist", relation=relation, storage_level=storageLevel
   2102     )
   2103     retur